# Analyse grondwaterstanden en opbarstsituatie ARK

Zie Beemster e.a. (2022) fig.3.5, tabel 3.1, de boringen achterin en de tabel 6.4 met soortelijk gewicht van de grondlagen.

In [1]:
import os
import sys
from typing import Any
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import pickle
from PIL import Image
from tools.etc.etc import logo

from mf6lab.Projects.ARK_RWS.src.ARK_geotop import (
    CrossSectionDigitizer,
    ImagePicker,
    Dirs,
    )

print(sys.executable)

# --- Needed to make figure separate from the notebook and interactive
%matplotlib qt

NOTEBOOK_NAME = "Beemster22_analysis.ipynb"
class Pixel_transform():
    def __init__(self, pxl_ext, world_ext):
        self.pxl_ext = pxl_ext
        self.world_ext = world_ext

    def pixel_to_world(self, px, py):
        pxmin, pxmax, pymin, pymax = self.pxl_ext
        x_min, x_max, z_min, z_max = self.world_ext

        x = x_min + (px - pxmin) / (pxmax - pxmin) * (x_max - x_min)

        # --- note: image y increases downward → z increases upward
        z = z_max - (py - pymin) / (pymax - pymin) * (z_max - z_min)

        return x, z
 

# --- Seet the namespace for the relevant directories
dirs = Dirs()

# --- Pickling
def pickleto(var:Any, basename:str, parent:str=dirs.data):
    """Pickle var to os.path.join(dirs.data, basename)"""
    if not basename.endswith('.pkl'):
        basename += ".pkl"

    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'wb') as f:
        print(f"Pickled {basename} --> {parent}")        
        pickle.dump(var, f)

# --- Unpickling
def picklefrom(basename:str, parent:str=dirs.data)->Any:
    """Unpickle varname from os.path.join(parent, basename)"""
    if not basename.endswith(".pkl"):
        basename += ".pkl"
          
    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'rb') as f:
        print(f"Loaded {basename} <-- {parent}")        
        return pickle.load(f)

# --- Color for empty legend (empty voxel with geo_unit 'none')
WHITE_01 = np.array([1., 1., 1.])

loading mfpath.py
/Users/Theo/Development/python/mf6_tools/mf6lab/.venv/bin/python


# Get the geotop.pdf obtained from dinoloket.nl

When reading it with pdf2image you get actually two pdfs.
One has the actual cross section and the other the legend and a map.

In [2]:
# --- Show images one after the other

png_paths = glob(dirs.images + '/dwars*.png')

for i, png_file in enumerate(png_paths):    
    basename = os.path.basename(png_file)
    img = Image.open(png_file)
    
    img_arr = np.asarray(img)
            
    # --- Imshow uses 0-255 as color if dtype is ints and 0-1. when dtype is float
    # --- Just to show it, doesn't matter.
    
    # --- The first pdf page contains the x-section
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(basename)
    ax.set_title(f"{basename}")
    ax.imshow(img) 
    ax.plot()
    
    

In [3]:
png_paths

['/Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/images/dwarsprofGWS_fig35.png']

Get the last pdf file in the dirs.dino directory.
Show its properties.

## Use the image picker with the legend image to get the legend colors

Instantiate the ImagePicker with the image that holds the map with the legend color boxes.
Then zoom in and click the legend's color boxes in the normal order.
This yields the colors to compare those sampled in the actual cross section with.

Press ENTER to finish with the legend.

In [39]:
# ---Instantiate the picker with the image to pick from (image with the legend)
picker = ImagePicker(img_arr)

# --- Zoom before clicking (zoom into legend). The easiest way is the shift the figure
# --- to hit the top panel of the screen. The shifted window will go full screen automatically.

# --- Step 1: Pick legend colors one after the other in sequence. Press ENTER when done.
colors = picker.get_colors(n=-1)

# --- Special for legend colors Remove white points first to remove mistake clicks
legend_colors = colors # np.array([clr for clr in colors if not np.all(clr > 0.95)])
legend_colors

array([[0.        , 0.        , 0.        , 1.        ],
       [0.75294118, 0.31372549, 0.30196078, 1.        ],
       [0.29411765, 0.6745098 , 0.77647059, 1.        ],
       [0.43921569, 0.18823529, 0.62745098, 1.        ]])

## With the legend_colors now obtained get the pixel extent of the cross section

To get the pixel extent of the cross section:

1. Instantiate the ImagePicker once again, but now with the image of the actual cross section.
2. Zoom in the same way has before. Preferably by shifting the image window to the top of the screen to let it go full screen.
3. Then pick the corners of the cross sections of which you know the world coordinates.
4. Press ENTER when finished. The pixel_exent is thus obtained.

In fact, a pixel bounding box is computed around all picked points and turned into the
pixel_extent: (pxmin, pxmax, pymin, pymax)


In [6]:
# --- Initiate a new picker, now with the cross section   
# --- To get the pixel bounding box
picker = ImagePicker(img_arr)

# --- Again, zoom in
# --- In fact you can click any number of points, the bbox uses there min and max coords
pxl_extent = picker.get_pxl_bbox(n=-1)

print("Pixel extent of the cross section:")
print("==================================")
print("[pxmin pxmax pymin pymax]")
print(np.array(pxl_extent))


Points picked in pixels:  [(101, 759), (1223, 764)]
Pixel extent of the cross section:
[pxmin pxmax pymin pymax]
[ 101 1223  759  764]


# Specific for fig.3.5 from Beemster e.a. (2022)

In [ ]:
# pixel and world extent of the axes of the cross section in Beemster e.a. 2022, fig. 3.5
# Gemiddelde grondwaterstanden en stijghoogten. Juli 2020 - Feb 201 in  polders BBO (links) en HGP (rechts)
pxl_extent = [103, 1125, 94, 760]
world_extent = [129300, 130200, -3.00, 1.00]

# --- Get the coordinates of the graphs by clicking in the image
PT = Pixel_transform(pxl_extent, world_extent)

# --- Names of lines to be clicked in sequenced. L = left, R = Right, k = kanaal,
#     phi = stijghoogte, gws = grondwaterstand, pp = polder peil.
line_names = ['pkL', 'pkR', 'phiL', 'phiR', 'gwsL', 'gwsR', 'ppL', 'ppR']

lines= {}
PP = ImagePicker(img_arr)
for ln in line_names:
    print(ln)
    # --- click the points for the current line
    px, py = np.array(PP.pick_points(n=-1)).T
    
    # --- convert to the world coordinates in the image
    x, z = PT.pixel_to_world(px, py)
    lines[ln] = {'x':x, 'z': z}

# --- Add line colors (use picked legend colors)
lines['kanaalpeil'] = {'x':np.array([129700., 129800.]),
                       'z':np.array([-0.4, -0.4]),
                       'color': 'blue'} 
    
print(lines)

pickleto(lines, "dsnfig_35_beemster.pkl")

pkL


In [62]:
lines = picklefrom("dsnfig_35_beemster.pkl")

Loaded dsnfig_35_beemster.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


In [115]:
# --- Add legend color to each line
clrs = []
for c in list(legend_colors):
    clrs.append(c)
    clrs.append(c)
for c, (k, item) in zip(clrs, lines.items()):
    item['color'] = c

labels = ['maaiveld', '', 'gem. stijghoogte', '', 'gem. gws', '', 'polderpeil', '']
# --- Plot the line in the x-section, verify with the original figure
#     Then we can use the world coordinates for analysis   
fig, ax = plt.subplots(figsize=(10, 6))
for label, (ln, item) in zip(labels, lines.items()):
    if ln.endswith('L'):
        ax.plot(item['x'], item['z'], marker='.', ms=4, color=item['color'], label=label)
    else:
        ax.plot(item['x'], item['z'], marker='.', ms=4, color=item['color'])

item = lines['kanaalpeil']
ax.plot(item['x'], item['z'], '-', color=item['color'], label='kanaalpeil' )

pbuis = {
'K09-73': {'x': 130107, 'phi': -1.57, 'bkb': -1.279, 'mv': -1.712, 'bkf': 3.24, 'ddkl': 2.00, 'klei': 0.00, 'veen': 2.00},
'K09-74': {'x': 129948, 'phi': -1.43, 'bkb': -1.265, 'mv': -1.662, 'bkf': 3.24, 'ddkl': 3.20, 'klei': 0.20, 'veen': 3.00},
'K09-75': {'x': 129855, 'phi': -1.36, 'bkb': -1.133, 'mv': -1.570, 'bkf': 3.24, 'ddkl': 2.90, 'klei': 0.35, 'veen': 2.55},
'L09-22': {'x': 129672, 'phi': -1.51, 'bkb': -1.308, 'mv': -1.873, 'bkf': 3.24, 'ddkl': 3.10, 'klei': 0.00, 'veen': 3.10},
'L09-23': {'x': 129572, 'phi': -1.52, 'bkb': -1.438, 'mv': -1.941, 'bkf': 3.24, 'ddkl': 3.65, 'klei': 0.40, 'veen': 3.25},
'L09-24': {'x': 129336, 'phi': -1.74, 'bkb': -1.531, 'mv': -2.059, 'bkf': 3.24, 'ddkl': 3.80, 'klei': 0.00, 'veen': 3.80},
}

rhoklei = 1.40
rhoveen = [0.98, 1.05, 1.10]
print(f"{'pbuis':>6} {'h[m]':>6} {'p1[m]':>6} {'p2[m]':>6} {'p3[m]':>6} {'phi[NAP]':>8} {'mv[NAP]':>8} {'zdek[NAP]':>8} {'klei[m]':>8} {'veen[m]':>8}")
print(f"{'':6} {'':6} {rhoveen[0]:6.2f} {rhoveen[1]:6.2f} {rhoveen[2]:6.2f}")
print(f"{'-------'}{'-------'}{'-------'}{'-------'}{'-------'}{'---------'}{'---------'}{'---------'}{'---------'}{'---------'}")

for p, item in pbuis.items():
    ax.text(item['x'], -2.95, f"{p}", rotation=90, ha='center')
    ax.plot(item['x'], item['mv'], 'ko', ms=5)
    item['zdekl'] = item['mv'] - item['klei'] - item['veen']
    h = item['phi'] - item['zdekl']
    p1 = item['klei'] * rhoklei + item['veen'] * rhoveen[0]
    p2 = item['klei'] * rhoklei + item['veen'] * rhoveen[1]
    p3 = item['klei'] * rhoklei + item['veen'] * rhoveen[2]
    print(f"{p} {h:6.2f} {p1:6.2f} {p2:6.2f} {p3:6.2f} {item['phi']:8.2f} {item['mv']:8.2f} {item['zdekl']:8.2f} {item['klei']:8.2f} {item['veen']:8.2f}")
    


 pbuis   h[m]  p1[m]  p2[m]  p3[m] phi[NAP]  mv[NAP] zdek[NAP]  klei[m]  veen[m]
                0.98   1.05   1.10
--------------------------------------------------------------------------------
K09-73   2.14   1.96   2.10   2.20    -1.57    -1.71    -3.71     0.00     2.00
K09-74   3.43   3.22   3.43   3.58    -1.43    -1.66    -4.86     0.20     3.00
K09-75   3.11   2.99   3.17   3.29    -1.36    -1.57    -4.47     0.35     2.55
L09-22   3.46   3.04   3.26   3.41    -1.51    -1.87    -4.97     0.00     3.10
L09-23   4.07   3.75   3.97   4.13    -1.52    -1.94    -5.59     0.40     3.25
L09-24   4.12   3.72   3.99   4.18    -1.74    -2.06    -5.86     0.00     3.80


In [96]:
lines['phiR']
lines['phiL']

{'x': array([129337.70053476, 129571.92513369, 129672.19251337]),
 'z': array([-1.74474474, -1.51651652, -1.51051051]),
 'color': array([0.75294118, 0.31372549, 0.30196078, 1.        ])}

In [107]:
dphi = lines['phiL']['z'] - lines['ppL']['z'][0]
x_ = np.linspace(0, 400)
lam = 1000.
dphiL0 = lines['phiL']['z'][-1] - lines['ppL']['z'][-1]
phiL   = lines['ppL']['z'][0] + dphiL0 * np.exp(-x_ / lam)
xL     = lines['phiL']['x'][-1] - x_

dphiR0 = lines['phiR']['z'][0] - lines['ppR']['z'][0]
phiR   = lines['ppR']['z'][0] + dphiR0 * np.exp(-x_ / lam)
xR     = lines['phiR']['x'][0] + x_

ax.plot(xL, phiL, 'r--', label=rf'$\phi_W$  $\lambda$={lam} m')
ax.plot(xR, phiR, 'r--', label=rf'$\phi_E$, $\lambda$={lam} m')

ax.set_xlabel('X-coördinaat')
ax.set_ylabel('m tov NAP')
ax.set_title(('gemiddelde grondwaterstanden en stijghoogten\n' +
              'juli 2020 - februari 2021 nabij ARK\n' +
              'links (W) polder BBO, rechts (E), polder HG'))
ax.grid()
ax.legend()
logo(fig, NOTEBOOK_NAME)
plt.show()

fig.savefig(os.path.join(dirs.images, "dsnfig_35_beemster.pdf"))